# RAG-Based Injury Feedback System
**Real-Time Workout Pose Estimation and Feedback System**  
MSc Research — M.K. Sathsara Rasantha

## Notebook Overview
This notebook implements a Retrieval-Augmented Generation (RAG) pipeline that:
1. Parses a curated biomechanical knowledge base into searchable chunks
2. Embeds chunks using `sentence-transformers` and stores in ChromaDB
3. Retrieves relevant guidance for a detected injury-prone pose
4. Generates specific corrective feedback using the Anthropic Claude API

**Input:** `phase_labeled.csv` — output from the phase-aware injury detection notebook  
**Output:** Per-row corrective feedback with severity, coaching cue, and citation

---
## Cell 1 — Install Dependencies
Run once. Restart the kernel after installation.

In [1]:
# Run once — restart kernel after this cell completes
%pip install sentence-transformers chromadb anthropic pandas --quiet

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\user\\Desktop\\Learning\\MSc\\Research\\MotionGuard\\venv\\Lib\\site-packages\\google\\~upb\\_message.pyd'
Check the permissions.



---
## Cell 2 — Imports and Configuration

In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
# Update these to match your local directory structure
KB_DIR      = "../knowledge_base"   # folder containing squat.txt, pull_up.txt etc.
DB_DIR      = "../chroma_db"        # ChromaDB will be created here
LABELED_CSV = "./phase_labeled.csv" # output from injury detection notebook
OUTPUT_CSV  = "./feedback_results.csv"

# ── Anthropic API key ────────────────────────────────────────────────────────
os.environ["ANTHROPIC_API_KEY"] = "anthropic_api_key_here"  # replace with your key

# ── RAG settings ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # local model, no API key needed
TOP_K           = 3                    # number of chunks to retrieve per query
CLAUDE_MODEL    = "claude-sonnet-4-6"

print("Configuration loaded.")
print(f"  Knowledge base : {KB_DIR}")
print(f"  ChromaDB       : {DB_DIR}")
print(f"  Labeled CSV    : {LABELED_CSV}")

Configuration loaded.
  Knowledge base : ../knowledge_base
  ChromaDB       : ../chroma_db
  Labeled CSV    : ./phase_labeled.csv


---
## Cell 3 — Parse Knowledge Base Files into Chunks

Reads each `.txt` file in the knowledge base folder and splits into one chunk per error.  
Each chunk carries: `exercise`, `phase`, `error_name`, `angle`, `severity`, `source`, `cue`.

In [8]:
def _extract_angle_column(angle_indicator_str):
    """Map angle indicator text to the CSV column name."""
    s = angle_indicator_str.lower()
    if "knee"     in s: return "knee_angles"
    if "elbow"    in s: return "elbow_angles"
    if "hip"      in s: return "hip_angles"
    if "shoulder" in s: return "shoulder_angles"
    return "unknown"


def parse_knowledge_base(kb_dir):
    """
    Parse all .txt knowledge base files into structured chunk dicts.
    Splits on === phase dividers, then on --- ERROR --- boundaries.
    """
    chunks = []
    kb_path = Path(kb_dir)

    for txt_file in sorted(kb_path.glob("*.txt")):
        # Derive exercise name from filename
        exercise = txt_file.stem.replace("_", " ")   # t_bar_row → t bar row
        raw = txt_file.read_text(encoding="utf-8")

        # Split into phase blocks on === dividers
        phase_blocks = re.split(r"={10,}", raw)

        current_phase        = None
        current_phase_desc   = None
        current_correct_form = None

        for block in phase_blocks:
            block = block.strip()
            if not block:
                continue

            # Phase header block — extract metadata, do not create a chunk
            phase_match = re.search(r"^PHASE:\s*(.+)$", block, re.MULTILINE)
            if phase_match:
                current_phase = phase_match.group(1).strip()

                desc_match = re.search(r"^DESCRIPTION:\s*(.+)$", block, re.MULTILINE)
                current_phase_desc = desc_match.group(1).strip() if desc_match else ""

                form_match = re.search(
                    r"CORRECT FORM:\s*\n(.*?)(?=\nANGLE RANGES:|$)",
                    block, re.DOTALL
                )
                current_correct_form = form_match.group(1).strip() if form_match else ""
                continue

            # Error block — split on --- ERROR N --- markers
            error_blocks = re.split(r"---\s*ERROR\s*\d+\s*---", block)

            for error_block in error_blocks:
                error_block = error_block.strip()
                if not error_block or not current_phase:
                    continue

                name_match = re.search(r"^NAME:\s*(.+)$", error_block, re.MULTILINE)
                if not name_match:
                    continue   # not a valid error block

                error_name = name_match.group(1).strip()

                # Extract structured fields
                angle_match    = re.search(r"^ANGLE INDICATOR:\s*(.+)$", error_block, re.MULTILINE)
                severity_match = re.search(r"^SEVERITY:\s*(.+)$",         error_block, re.MULTILINE)
                source_match   = re.search(r"^SOURCE:\s*(.+)$",           error_block, re.MULTILINE)
                mechanism_match = re.search(
                    r"MECHANISM:\s*\n(.*?)(?=\nSOURCE:|$)", error_block, re.DOTALL
                )
                cue_match = re.search(
                    r"CORRECTION CUE:\s*\n(.*?)(?=\n---|$)", error_block, re.DOTALL
                )

                angle_indicator = angle_match.group(1).strip()     if angle_match     else ""
                severity        = severity_match.group(1).strip()  if severity_match  else "warning"
                source          = source_match.group(1).strip()    if source_match    else ""
                mechanism       = mechanism_match.group(1).strip() if mechanism_match else ""
                cue_full        = cue_match.group(1).strip()       if cue_match       else ""

                cue_line_match = re.search(r'Cue:\s*"(.+?)"', cue_full)
                short_cue = cue_line_match.group(1) if cue_line_match else cue_full[:100]

                angle_col = _extract_angle_column(angle_indicator)

                # Build chunk text — this is what gets embedded and retrieved
                chunk_text = f"""Exercise: {exercise}
Phase: {current_phase}
Phase description: {current_phase_desc}
Error: {error_name}
Angle indicator: {angle_indicator}
Severity: {severity}
Correct form context: {current_correct_form}
Biomechanical mechanism: {mechanism}
Corrective guidance: {cue_full}
Source: {source}""".strip()

                chunks.append({
                    "text":        chunk_text,
                    "exercise":    exercise,
                    "phase":       current_phase,
                    "error_name":  error_name,
                    "angle":       angle_col,
                    "severity":    severity,
                    "source":      source,
                    "short_cue":   short_cue,
                    "mechanism":   mechanism,
                    "cue_full":    cue_full,
                })

    return chunks


# Run parsing
chunks = parse_knowledge_base(KB_DIR)

print(f"Total chunks parsed: {len(chunks)}")
print()
for ex in sorted(set(c["exercise"] for c in chunks)):
    ex_chunks = [c for c in chunks if c["exercise"] == ex]
    phases    = sorted(set(c["phase"] for c in ex_chunks))
    print(f"  {ex:20s} → {len(ex_chunks)} chunks across phases: {phases}")

Total chunks parsed: 21

  leg extension        → 3 chunks across phases: ['extended', 'start_bent']
  pull up              → 4 chunks across phases: ['dead_hang', 'top_position']
  shoulder press       → 4 chunks across phases: ['lockout', 'rack_position']
  squat                → 5 chunks across phases: ['bottom_position', 'standing']
  t bar row            → 5 chunks across phases: ['full_contraction', 'start_position']


---
## Cell 4 — Inspect a Sample Chunk
Verify the parsing output looks correct before embedding.

In [9]:
# Inspect a single chunk to verify structure
sample = chunks[0]

print("=" * 60)
print("SAMPLE CHUNK")
print("=" * 60)
print(f"exercise   : {sample['exercise']}")
print(f"phase      : {sample['phase']}")
print(f"error_name : {sample['error_name']}")
print(f"angle      : {sample['angle']}")
print(f"severity   : {sample['severity']}")
print(f"source     : {sample['source']}")
print(f"short_cue  : {sample['short_cue']}")
print()
print("CHUNK TEXT (what gets embedded):")
print("-" * 60)
print(sample['text'])

SAMPLE CHUNK
exercise   : leg extension
phase      : start_bent
error_name : Excessive knee flexion at start — knee over-bent
angle      : knee_angles
severity   : warning
source     : Steinkamp et al. (1993) AJSM 21(3):438–444; Escamilla et al. (1998) MSSE 30(4):556–569
short_cue  : Start with a 90° bend — not deeper. Adjust the seat if needed.

CHUNK TEXT (what gets embedded):
------------------------------------------------------------
Exercise: leg extension
Phase: start_bent
Phase description: Starting position with the knee flexed, foot behind the pad.
Error: Excessive knee flexion at start — knee over-bent
Angle indicator: knee_angle < 70°
Severity: warning
Correct form context: At the starting position, the knee angle should fall between 70° and 109°,
  reflecting a comfortable starting flexion that allows controlled extension
  through the full range. Steinkamp et al. (1993) established that patellofemoral
  compressive forces are elevated at knee flexion angles between 60° an

---
## Cell 5 — Embed Chunks and Build ChromaDB Vector Store

Uses `all-MiniLM-L6-v2` — a lightweight local embedding model (no API key needed).  
Stores vectors in ChromaDB with exercise/phase metadata for filtered retrieval.  
**Run once.** The database persists to disk at `DB_DIR`.

In [5]:
%pip install sentence-transformers chromadb

  Using cached sentence_transformers-5.6.0-py3-none-any.whl (596 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl (23.5 MB)
  Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl (693 kB)
  Using cached torch-2.12.0-cp311-cp311-win_amd64.whl (123.0 MB)
  Using cached build-1.5.0-py3-none-any.whl (26 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl (60 kB)
  Using cached uvicorn-0.49.0-py3-none-any.whl (71 kB)
  Using cached onnxruntime-1.27.0-cp311-cp311-win_amd64.whl (13.4 MB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl (19 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl (2.8 MB)
  Using cached overrides-7.7.0-py3-none-any.whl (17 kB)
  Using cached importlib_resources-7.

In [12]:
from sentence_transformers import SentenceTransformer
import chromadb

print(f"Loading embedding model: {EMBEDDING_MODEL}")
embed_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Embedding {len(chunks)} chunks...")
texts      = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True).tolist()

# Initialise persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path=DB_DIR)

# Delete and recreate collection for a clean build
try:
    chroma_client.delete_collection("exercise_kb")
    print("Deleted existing collection.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="exercise_kb",
    metadata={"hnsw:space": "cosine"}   # cosine similarity for text vectors
)

# Add all chunks with metadata
collection.add(
    ids        = [f"chunk_{i}" for i in range(len(chunks))],
    embeddings = embeddings,
    documents  = texts,
    metadatas  = [
        {
            "exercise":   c["exercise"],
            "phase":      c["phase"],
            "angle":      c["angle"],
            "severity":   c["severity"],
            "error_name": c["error_name"],
            "source":     c["source"],
            "short_cue":  c["short_cue"],
            "mechanism":  c["mechanism"][:500],   # ChromaDB metadata char limit
        }
        for c in chunks
    ]
)

print(f"\nVector store built successfully.")
print(f"  Location      : {DB_DIR}")
print(f"  Total vectors : {collection.count()}")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6217.90it/s]


Embedding 21 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]



Vector store built successfully.
  Location      : ../chroma_db
  Total vectors : 21


---
## Cell 6 — Define Retrieval Functions

Two functions:  
- `build_query()` — converts injury detection output into a semantic search string  
- `retrieve_chunks()` — queries ChromaDB, filtered by exercise + phase

In [13]:
def build_query(exercise, phase, violated_rules, angles=None):
    """
    Build a semantic query string from injury detection output.
    This is passed to the embedding model and used to find
    the most relevant chunks in the vector store.
    """
    angle_str = ""
    if angles:
        parts = [f"{k}={v:.1f}°" for k, v in angles.items() if v is not None]
        angle_str = ", ".join(parts)

    query = f"""Exercise: {exercise}
Phase: {phase}
Detected violations: {violated_rules}
Angle values: {angle_str}
What is the biomechanical risk and how should the athlete correct their form?"""

    return query.strip()


def retrieve_chunks(query, exercise, phase, top_k=TOP_K):
    """
    Retrieve the top_k most semantically similar chunks from ChromaDB.
    Filters by exercise AND phase before semantic ranking — ensures
    retrieved chunks are always exercise/phase specific.
    """
    query_embedding = embed_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = min(top_k, collection.count()),
        where            = {
            "$and": [
                {"exercise": {"$eq": exercise}},
                {"phase":    {"$eq": phase}}
            ]
        },
        include = ["documents", "metadatas", "distances"]
    )

    retrieved = []
    if results["documents"] and results["documents"][0]:
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            retrieved.append({
                "text":       doc,
                "metadata":   meta,
                "distance":   dist,
                "similarity": round(1 - dist, 3)   # cosine: 1=identical, 0=unrelated
            })

    return retrieved


print("Retrieval functions defined.")

Retrieval functions defined.


---
## Cell 7 — Test Retrieval on a Sample Query

Verify that the vector store retrieves the correct chunks before connecting the LLM.  
Check that `similarity` scores are reasonable (> 0.5 is a good sign).

In [14]:
# Test retrieval with a known injury case
test_exercise = "squat"
test_phase    = "bottom_position"
test_violated = "knee_angle=52.0° outside [55°–100°]: patellar compressive stress"
test_angles   = {"knee": 52.0, "hip": 88.3}

test_query = build_query(test_exercise, test_phase, test_violated, test_angles)
print("QUERY:")
print(test_query)
print()

retrieved = retrieve_chunks(test_query, test_exercise, test_phase)

print(f"Retrieved {len(retrieved)} chunks:")
print("=" * 60)
for i, chunk in enumerate(retrieved, 1):
    print(f"\n[{i}] similarity = {chunk['similarity']}")
    print(f"     error      : {chunk['metadata']['error_name']}")
    print(f"     severity   : {chunk['metadata']['severity']}")
    print(f"     source     : {chunk['metadata']['source'][:80]}")
    print(f"     cue        : {chunk['metadata']['short_cue']}")

QUERY:
Exercise: squat
Phase: bottom_position
Detected violations: knee_angle=52.0° outside [55°–100°]: patellar compressive stress
Angle values: knee=52.0°, hip=88.3°
What is the biomechanical risk and how should the athlete correct their form?

Retrieved 3 chunks:

[1] similarity = 0.683
     error      : Excessive knee flexion depth
     severity   : warning
     source     : Escamilla (2001) Med. Sci. Sports Exerc. 33(1):127–141
     cue        : Squat to parallel — depth means nothing if your back rounds.

[2] similarity = 0.645
     error      : Forward lean / torso collapse
     severity   : warning
     source     : Myer et al. (2014) SCJ 36(6):4–27; McGill (2015) Back Mechanic
     cue        : Lead with your chest, not your hips.

[3] similarity = 0.604
     error      : Butt wink (posterior pelvic tilt / lumbar flexion)
     severity   : danger
     source     : McGill (2015) Back Mechanic; Schoenfeld (2010) JSCR 24(12):3497–3506
     cue        : Sit tall into the hole — ch

---
## Cell 8 — Define Feedback Generation Function

Passes retrieved chunks as grounded context to Claude.  
The LLM is instructed to generate feedback based **only** on the retrieved material —  
this prevents hallucination and keeps all feedback traceable to cited sources.

In [8]:
%pip install anthropic

  Using cached anthropic-0.109.2-py3-none-any.whl (923 kB)
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached docstring_parser-0.18.0-py3-none-any.whl (22 kB)
  Using cached jiter-0.15.0-cp311-cp311-win_amd64.whl (200 kB)
Note: you may need to restart the kernel to use updated packages.


In [15]:
import anthropic

claude_client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment


def generate_feedback(exercise, phase, violated_rules, angles, retrieved_chunks):
    """
    Generate corrective feedback using Claude with retrieved chunks as context.

    Returns a dict with:
        severity : "warning" or "danger"
        summary  : one-sentence problem description
        feedback : 2-3 sentences of specific corrective guidance
        cue      : short coaching cue (under 10 words)
        source   : citation from the knowledge base
    """
    # Build context block from retrieved chunks
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[Reference {i} | similarity: {chunk['similarity']}]\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Build angle string for the prompt
    angle_str = ""
    if angles:
        angle_str = ", ".join(
            f"{k}: {v:.1f}°" for k, v in angles.items() if v is not None
        )

    # Severity from retrieved chunks — escalate to danger if any chunk is danger
    severities = [c["metadata"].get("severity", "warning") for c in retrieved_chunks]
    severity   = "danger" if "danger" in severities else "warning"

    prompt = f"""You are an expert exercise physiotherapist and strength coach.
A pose estimation system has detected an injury-prone exercise position.

DETECTED SITUATION:
- Exercise: {exercise}
- Phase: {phase}
- Violations: {violated_rules}
- Measured angles: {angle_str}

BIOMECHANICAL REFERENCE MATERIAL:
{context}

Based ONLY on the reference material above, provide corrective feedback.
Respond in JSON with exactly these fields:
{{
  "severity": "{severity}",
  "summary": "One sentence describing the main problem detected",
  "feedback": "2-3 sentences of specific corrective feedback referencing the actual angle values",
  "cue": "One short coaching cue the athlete can remember (under 10 words)",
  "source": "The citation(s) from the reference material that support this feedback"
}}

Rules:
- Always reference the specific angle values in the feedback
- Do not invent information not in the reference material
- Keep the cue under 10 words
- Be direct and actionable, not generic
Respond with valid JSON only."""

    response = claude_client.messages.create(
        model      = CLAUDE_MODEL,
        max_tokens = 1000,
        messages   = [{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text.strip()

    try:
        clean  = re.sub(r"```json|```", "", raw).strip()
        result = json.loads(clean)
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails
        result = {
            "severity": severity,
            "summary":  f"Injury-prone position detected in {exercise} at {phase}",
            "feedback": raw[:300],
            "cue":      "Check your form",
            "source":   retrieved_chunks[0]["metadata"].get("source", "") if retrieved_chunks else ""
        }

    return result


print("Feedback generation function defined.")

Feedback generation function defined.


---
## Cell 9 — End-to-End Pipeline Function

Wraps all steps into two convenience functions:  
- `get_feedback()` — takes raw values  
- `get_feedback_from_row()` — takes a dataframe row directly

In [16]:
def get_feedback(exercise, phase, violated_rules, angles=None):
    """
    Full RAG pipeline: build query → retrieve → generate.

    Args:
        exercise      : e.g. "squat"
        phase         : e.g. "bottom_position"
        violated_rules: violation string from phase_labeled.csv
        angles        : dict e.g. {"knee": 52.0, "hip": 88.0}

    Returns:
        dict: severity, summary, feedback, cue, source
    """
    query  = build_query(exercise, phase, violated_rules, angles)
    chunks = retrieve_chunks(query, exercise, phase)

    if not chunks:
        return {
            "severity": "warning",
            "summary":  f"Injury-prone position detected: {exercise} / {phase}",
            "feedback": f"Violation: {violated_rules}. Please review your form.",
            "cue":      "Check your form",
            "source":   ""
        }

    return generate_feedback(exercise, phase, violated_rules, angles, chunks)


def get_feedback_from_row(row):
    """
    Convenience wrapper — pass a row from phase_labeled.csv directly.
    Extracts exercise, phase, violated_rules and angle values automatically.
    """
    exercise       = row.get("exercise_key", "")
    phase          = row.get("phase", "")
    violated_rules = row.get("violated_rules", "")

    # Extract available angle values from the row
    angles = {}
    for col, key in [
        ("elbow_angles",    "elbow"),
        ("hip_angles",      "hip"),
        ("knee_angles",     "knee"),
        ("shoulder_angles", "shoulder")
    ]:
        val = row.get(col)
        try:
            angles[key] = float(val)
        except (TypeError, ValueError):
            pass

    return get_feedback(exercise, phase, violated_rules, angles)


print("Pipeline functions defined.")

Pipeline functions defined.


---
## Cell 10 — Test Full Pipeline on a Single Example

Run the complete pipeline on one hardcoded example before running on the full dataset.  
Verify the output looks correct and the source citations are present.

In [17]:
# Single end-to-end test
result = get_feedback(
    exercise       = "squat",
    phase          = "bottom_position",
    violated_rules = "knee_angle=52.0° outside [55°–100°]: patellar compressive stress",
    angles         = {"knee": 52.0, "hip": 88.3}
)

print("=" * 60)
print("FEEDBACK RESULT")
print("=" * 60)
print(f"Severity : {result['severity']}")
print(f"Summary  : {result['summary']}")
print(f"Feedback : {result['feedback']}")
print(f"Cue      : {result['cue']}")
print(f"Source   : {result['source']}")

FEEDBACK RESULT
Severity : warning
Summary  : Excessive knee flexion depth at 52° is placing dangerous compressive stress on the patellofemoral joint and menisci.
Feedback : Your knee angle of 52° falls below the safe minimum of 55°, significantly increasing patellofemoral and tibiofemoral compressive forces according to Escamilla (2001). Your hip angle of 88.3° is within the acceptable range, so the primary correction needed is to reduce squat depth until your knee angle stays at or above 55°. For most athletes, squatting to parallel (approximately 90° knee angle) provides sufficient training stimulus without excessive joint stress.
Cue      : Squat to parallel — depth means nothing if it hurts.
Source   : Escamilla (2001) Med. Sci. Sports Exerc. 33(1):127–141; Steinkamp et al. (1993)


---
## Cell 11 — Load Labeled Dataset and Preview Injury-Prone Rows

In [18]:
# Load the phase-aware labeled CSV from the injury detection notebook
df = pd.read_csv(LABELED_CSV)

# Cast angle columns to float (handles "Not Found" strings)
for col in ["shoulder_angles", "elbow_angles", "hip_angles", "knee_angles"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

injury_rows = df[df["label"] == "injury_prone"].reset_index(drop=True)

print(f"Total rows in dataset : {len(df)}")
print(f"Injury-prone rows     : {len(injury_rows)}")
print()
print("Injury-prone distribution per exercise:")
print(injury_rows["exercise_key"].value_counts().to_string())
print()
injury_rows[["exercise_key", "phase", "violated_rules",
             "elbow_angles", "hip_angles", "knee_angles"]].head(10)

Total rows in dataset : 4
Injury-prone rows     : 4

Injury-prone distribution per exercise:
exercise_key
t bar row    4



,exercise_key,phase,violated_rules,elbow_angles,hip_angles,knee_angles
0,t bar row,start_position,hip_angles=166.7° outside [80°–145°]: Hip hing...,174.040468,166.662446,NaN
1,t bar row,start_position,hip_angles=178.9° outside [80°–145°]: Hip hing...,170.348988,178.934198,179.006823
2,t bar row,start_position,hip_angles=170.6° outside [80°–145°]: Hip hing...,166.528553,170.610122,NaN
3,t bar row,start_position,hip_angles=165.4° outside [80°–145°]: Hip hing...,170.284530,165.440174,NaN


---
## Cell 12 — Run Feedback Generation on All Injury-Prone Rows

Iterates over every `injury_prone` row and generates feedback.  
Results are stored in a list and saved to `feedback_results.csv`.  

> **Note:** Each row makes one Claude API call. With ~70 injury rows this takes ~2–3 minutes.
> A progress counter is printed every 5 rows.

In [19]:
feedback_records = []

for idx, row in injury_rows.iterrows():
    try:
        result = get_feedback_from_row(row)

        feedback_records.append({
            # Original columns
            "image_path":     row.get("image_path", ""),
            "exercise_key":   row.get("exercise_key", ""),
            "phase":          row.get("phase", ""),
            "violated_rules": row.get("violated_rules", ""),
            "elbow_angles":   row.get("elbow_angles"),
            "hip_angles":     row.get("hip_angles"),
            "knee_angles":    row.get("knee_angles"),
            # Generated feedback
            "severity":       result.get("severity", ""),
            "summary":        result.get("summary", ""),
            "feedback":       result.get("feedback", ""),
            "cue":            result.get("cue", ""),
            "source":         result.get("source", ""),
        })

        if (idx + 1) % 5 == 0 or idx == 0:
            print(f"  [{idx + 1}/{len(injury_rows)}] {row.get('exercise_key')} / {row.get('phase')} → {result.get('severity')}")

    except Exception as e:
        print(f"  [{idx + 1}] ERROR: {e}")
        feedback_records.append({
            "image_path":     row.get("image_path", ""),
            "exercise_key":   row.get("exercise_key", ""),
            "phase":          row.get("phase", ""),
            "violated_rules": row.get("violated_rules", ""),
            "elbow_angles":   row.get("elbow_angles"),
            "hip_angles":     row.get("hip_angles"),
            "knee_angles":    row.get("knee_angles"),
            "severity": "error", "summary": str(e),
            "feedback": "", "cue": "", "source": ""
        })

feedback_df = pd.DataFrame(feedback_records)
feedback_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nDone. {len(feedback_records)} rows processed.")
print(f"Saved → {OUTPUT_CSV}")

  [1/4] t bar row / start_position → danger

Done. 4 rows processed.
Saved → ./feedback_results.csv


---
## Cell 13 — Review Results Summary

In [20]:
print("=" * 60)
print("FEEDBACK GENERATION SUMMARY")
print("=" * 60)
print(f"Total feedback generated : {len(feedback_df)}")
print()

print("Severity distribution:")
print(feedback_df["severity"].value_counts().to_string())
print()

print("Per-exercise breakdown:")
print(feedback_df.groupby(["exercise_key", "severity"]).size().to_string())

feedback_df[["exercise_key", "phase", "severity", "summary", "cue"]].head(10)

FEEDBACK GENERATION SUMMARY
Total feedback generated : 4

Severity distribution:
severity
danger    4

Per-exercise breakdown:
exercise_key  severity
t bar row     danger      4


,exercise_key,phase,severity,summary,cue
0,t bar row,start_position,danger,The hip angle of 166.7° indicates the torso is...,"Hinge — push hips back, chest forward, torso p..."
1,t bar row,start_position,danger,The hip angle of 178.9° indicates the torso is...,"Hinge — push hips back, chest forward, torso p..."
2,t bar row,start_position,danger,The hip angle of 170.6° indicates the torso is...,"Hinge — push hips back, chest forward, torso p..."
3,t bar row,start_position,danger,The hip angle of 165.4° indicates the torso is...,"Hinge — push hips back, chest forward, torso p..."


---
## Cell 14 — Inspect Sample Feedback Outputs Per Exercise

Print one detailed example per exercise for thesis documentation.

In [21]:
for exercise in feedback_df["exercise_key"].dropna().unique():
    sample = feedback_df[feedback_df["exercise_key"] == exercise].iloc[0]

    print("=" * 60)
    print(f"EXERCISE  : {exercise.upper()}")
    print(f"Phase     : {sample['phase']}")
    print(f"Violation : {sample['violated_rules']}")
    print("-" * 60)
    print(f"Severity  : {sample['severity']}")
    print(f"Summary   : {sample['summary']}")
    print(f"Feedback  : {sample['feedback']}")
    print(f"Cue       : {sample['cue']}")
    print(f"Source    : {sample['source']}")
    print()

EXERCISE  : T BAR ROW
Phase     : start_position
Violation : hip_angles=166.7° outside [80°–145°]: Hip hinge angle outside safe range — elevated L4/L5 spinal load
------------------------------------------------------------
Severity  : danger
Summary   : The hip angle of 166.7° indicates the torso is too upright at the start position, failing to establish the required hip hinge for a safe and effective T-bar row.
Feedback  : Your hip angle of 166.7° significantly exceeds the safe range of 80°–145°, meaning the torso is too upright before the pull begins. This shifts the bar path in front of your centre of mass and reduces lat and rhomboid activation, while also setting up poor mechanics throughout the set. Hinge forward at the hips — pushing the hips back and chest forward — until the torso is between 45° and parallel to the floor, maintaining a neutral lumbar curve throughout.
Cue       : Hinge — push hips back, chest forward, torso parallel.
Source    : Signorile et al. (2002) JSCR 1